In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [ ]:
# Check GPU status
!nvidia-smis

In [ ]:
%%capture
# For Qwen3-VL
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.57.1
%pip install --no-deps trl==0.22.2

# Configuration

In [ ]:
SEED = 42

# Model configuration
SFT_LORA_ID = 'alxxtexxr/Qwen3-VL-8B-Thinking-VCB8K-SFT-QLoRA-v260426081652'

# Data configuration
RL_DATA_ID = 'alxxtexxr/ViRL1.25K'

# Training configuration
MINI_BATCH_SIZE = 2
GRAD_ACC_STEPS = 4
NUM_EPOCHS = 1
WARMUP_STEPS = 5
LR = 5e-6

In [ ]:
from datetime import datetime

# Resume training configuration
RESUME_MODEL_ID = None
resume_from_checkpoint = bool(RESUME_MODEL_ID)
if resume_from_checkpoint:
    # hub_model_id = RESUME_MODEL_ID
    # project_name = hub_model_id.split('/')[-1]
    # model_name = project_name

    # from huggingface_hub import snapshot_download
    # snapshot_download(repo_id=hub_model_id, local_dir=model_name)
    
    raise NotImplementedError("Resuming from checkpoint is not implemented yet!")
else:
    model_name = SFT_LORA_ID
    project_name = f'{model_name.split("/")[-1].split('-SFT-QLoRA-v')[0]}-GRPO-QLoRA-v{datetime.now().strftime("%y%m%d%H%M%S")}'
    hub_model_id = f'alxxtexxr/{project_name}'
print("Resume from checkpoint:", resume_from_checkpoint)
print("Model name:", model_name)
print("Project name:", project_name)
print("Hugging Face model ID:", hub_model_id)

In [ ]:
import os
from unsloth import FastVisionModel
import random
import numpy as np
import torch
import transformers

def set_seed(seed, verbose=True):
    # Set random seed for os
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'  # CUDA deterministic behavior (11.2+)

    # Set random seed for NumPy
    np.random.seed(seed)

    # Set random seed for Torch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)           # If using multi-GPU
    # torch.use_deterministic_algorithms(True) # ERROR!
    torch.backends.cudnn.deterministic = True  # Ensures deterministic results
    torch.backends.cudnn.benchmark = False     # Avoids non-deterministic algorithms

    # Set random seed for Transformers
    transformers.set_seed(seed)

    # Set random seed for sklearn and Python's own random module
    random.seed(seed) 
    
    if verbose:
        print("Set random seed:", seed)

set_seed(SEED)

In [ ]:
# Fix TorchDynamo issue
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

# Fix Unsloth issue
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

# Model

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = SFT_LORA_ID,
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16-bit LoRA.
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # dtype = torch.float16, # Force FP16
)

# Data

In [ ]:
from datasets import load_dataset

dataset = load_dataset(RL_DATA_ID)
print(dataset)

In [ ]:
train_set = dataset['train']
val_set = dataset['validation']

print("Train set:")
print(train_set)
print("\nValidation set:")
print(val_set)

In [ ]:
# Resize to (512, 512)
def resize_images(example):
    image = example["image"]
    image = image.resize((512, 512))
    example["decoded_image"] = image
    return example
train_set = train_set.map(resize_images)
val_set = val_set.map(resize_images)

# Then convert to RGB
def convert_to_rgb(example):
    image = example["decoded_image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    example["decoded_image"] = image
    return example
train_set = train_set.map(convert_to_rgb)
val_set = val_set.map(convert_to_rgb)

In [ ]:
def clean_question(example):
    question = example["question"]
    question = question.replace("<image>", "")    # Remove "<image>" from the question text
    question = " ".join(question.strip().split()) # Remove extra whitespace
    example["question"] = question
    return example

train_set = train_set.map(clean_question)
val_set = val_set.map(clean_question)

In [ ]:
# Define the delimiter variables for clarity and easy modification
THINK_START = "<think>"
THINK_END = "</think>"
VTHINK_START = "<vthink>"
VTHINK_END = "</vthink>"
ANSWER_START = "<answer>"
ANSWER_END = "</answer>"

def make_conversation(example):
    # Define placeholder constants if they are not defined globally
    # The user's text prompt
    text_content = (
        f"Question: {example['question']}\n\n"
        # f"Please first reason step by step using visual representations between {VTHINK_START} and {VTHINK_END}, "
        # f"then reason step by step in natural language between {THINK_START} and {THINK_END}, "
        # "and finally provide your final answer in LaTeX using \\boxed{} "
        # f"between {ANSWER_START} and {ANSWER_END}"
        f"Please reason through the problem step by step, using visual representations between {VTHINK_START} and {VTHINK_END} "
        f"and natural language between {THINK_START} and {THINK_END}. "
        "Provide your final answer in LaTeX using \\boxed{} "
        f"between {ANSWER_START} and {ANSWER_END}."
    )

    # Construct the prompt in the desired multi-modal format
    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # Placeholder for the image
                {"type": "text", "text": text_content},  # The text part of the prompt
            ],
        },
    ]
    # The actual image data is kept separate for the processor
    return {"prompt": prompt, "image": example["decoded_image"], "answer": example["answer"]}

train_dataset = train_set.map(make_conversation)
val_dataset = val_set.map(make_conversation)

# We're reformatting dataset like this because decoded_images are the actual images
# The "image": example["decoded_image"] does not properly format the dataset correctly

# 1. Remove the original 'image' column
train_dataset = train_dataset.remove_columns("image")
val_dataset = val_dataset.remove_columns("image")

# 2. Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column("decoded_image", "image")
val_dataset = val_dataset.rename_column("decoded_image", "image")

In [ ]:
image = train_dataset[100]["image"]
prompt = train_dataset[100]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = False)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

# Reward Functions

In [ ]:
import re

THINK_PATTERN = re.compile(rf"{THINK_START}(.*?){THINK_END}", re.DOTALL)
VTHINK_PATTERN = re.compile(rf"{VTHINK_START}(.*?){VTHINK_END}", re.DOTALL)
ANSWER_PATTERN = re.compile(rf"{ANSWER_START}(.*?){ANSWER_END}", re.DOTALL)
VTOK_ID_PATTERN = re.compile(r'<\|vtok_(\d+)\|>')

def formatting_reward_func(completions, **kwargs) -> list[float]:
    LVL1_REWARD = 1.0
    LVL2_REWARD = 1.0
    LVL2_PENALTY = -0.9
    LVL1_PENALTY = -0.162
    TOTAL_REWARD = 6.0
    
    scores = []
    
    for i, completion in enumerate(completions):
        # print("[TEST] Completion:", i)
        score = 0
        
        if isinstance(completion, list):
            completion = completion[0]['content'] if completion else ""
        
        think_matches = THINK_PATTERN.findall(completion)
        vthink_matches = VTHINK_PATTERN.findall(completion)
        answer_matches = ANSWER_PATTERN.findall(completion)
        
        if len(answer_matches) == 1:
            # print("[TEST] Reward 1:", LVL1_REWARD)
            score += LVL1_REWARD
            # if answer_matches[0].strip().startswith('\\boxed{'):
            #     # print("[TEST] Reward 1.5:", LVL2_REWARD)
            #     score += LVL2_REWARD
            # else:
            #     # print("[TEST] Reward 1.5:", LVL2_PENALTY)
            #     score += LVL2_PENALTY
        else:
            # print("[TEST] Reward 1:", LVL1_PENALTY)
            score += LVL1_PENALTY
        
        if len(think_matches) > 0:
            # print("[TEST] Reward 2:", LVL1_REWARD)
            score += LVL1_REWARD
        else:
            # print("[TEST] Reward 2:", LVL1_PENALTY)
            score += LVL1_PENALTY
        
        vthink_count = len(vthink_matches)
        if vthink_count > 0:
            # print("[TEST] Reward 3:", LVL1_REWARD)
            score += LVL1_REWARD
            
            # Check if vthinking contains only valid visual tokens
            for vthink_match in vthink_matches:
                vthink_text = vthink_match.strip()
                
                # valid = all([0 <= int(i) <= 8191 if i.isdigit() else False for i in vthink_text.replace("<|vtok_", "").split("|>")][:-1])
                
                vtok_ids = VTOK_ID_PATTERN.findall(vthink_text)        # Extract all token IDs
                cleaned = VTOK_ID_PATTERN.sub("", vthink_text).strip() # Remove them, check remainder
                valid = (
                    bool(vtok_ids) and
                    all(0 <= int(i) <= 8191 for i in vtok_ids) and
                    not cleaned
                )
                
                if valid:
                    # print("[TEST] Reward 3.5:", LVL2_REWARD / vthink_count)
                    score += LVL2_REWARD / vthink_count
                else:
                    # print("[TEST] Reward 3.5:", LVL2_PENALTY / vthink_count)
                    score += LVL2_PENALTY / vthink_count
        else:
            # print("[TEST] Reward 4:", LVL1_PENALTY)
            score += LVL1_PENALTY
        
        # Check for visual tokens outside of vthinking
        outside_vthink_text = re.sub(f"{VTHINK_START}.*?{VTHINK_END}", "", completion, flags=re.DOTALL)
        if '<|vtok_' not in outside_vthink_text:
            # print("[TEST] Reward 4:", LVL1_REWARD)
            score += LVL1_REWARD
        else:
            # outside_vtok_count = len(VTOK_ID_PATTERN.findall(outside_vthink_text))
            # penalty_4 = max(LVL1_PENALTY * 0.1 * outside_vtok_count, LVL1_PENALTY) # Cap the penalty to LVL1_PENALTY
            penalty_4 = LVL1_PENALTY
            
            # print("[TEST] Reward 4:", penalty_4)
            score += penalty_4

        # Fix up addCriterion issues
        # See https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks
        # Penalize on excessive addCriterion and newlines
        if len(completion) != 0:
            removal = completion.replace("addCriterion", "").replace("\n", "")
            removal_ratio = (len(completion) - len(removal)) / len(completion)
            if removal_ratio < 0.5:
                # print("[TEST] Reward 7:", LVL1_REWARD)
                score += LVL1_REWARD
            else:
                # print("[TEST] Reward 7:", LVL1_PENALTY * removal_ratio)
                score += LVL1_PENALTY * removal_ratio # Penalize proportionally to the excessiveness of addCriterion/newlines
            
        # Normalize score; penalties are intentionally asymmetric —
        # individual penalties are kept mild to avoid canceling out earned rewards.
        # print("[TEST] Total score:", score)
        score /= TOTAL_REWARD

        scores.append(score)
        
        # print("[TEST] ================================\n")
        
    return scores

formatting_reward_func([
# 0. Good example
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<think>Hmm...</think>

<answer>\\boxed{Yes}</answer>""",

# 1. Bad example: non-visual tokens in vthinking
"""<vthink><|vtok_0|>Hello, world!<|vtok_8191|></vthink>

<think>
Some reasoning.
</think>

<answer>\\boxed{Yes}</answer>""",

# 2. Bad example: visual tokens exceeding the valid range
"""<vthink><|vtok_0|><|vtok_8192|></vthink>

<think>Hmm...</think>

<answer>\\boxed{Yes}</answer>""",

# 3. Bad example: visual tokens outside of vthinking
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<think>Hmm...</think>

<|vtok_1|><|vtok_2|>

<answer>\\boxed{Yes}</answer>""",

# # 4. Bad example: answer doesn't use \boxed{}
# """<vthink><|vtok_0|><|vtok_8191|></vthink>

# <think>Hmm...</think>

# <answer>Yes</answer>""",

# 4. Bad example: doesn't contain vthinking
"""<think>Hmm...</think>

<answer>\\boxed{Yes}</answer>""",

# 5. Bad example: doesn't contain thinking
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<answer>\\boxed{Yes}</answer>""",

# 6. Bad example: doesn't use answer tag
"""<vthink><|vtok_0|><|vtok_8191|></vthink>

<think>Hmm...</think>

\\boxed{Yes}""",
])

In [ ]:
def strip_boxed(answer: str) -> str:
    if "\\boxed{" in answer:
        start = answer.index("\\boxed{") + len("\\boxed{")
        depth = 1
        for i, c in enumerate(answer[start:], start):
            if c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    return answer[start:i]
    return answer

def standardize_answer(answer: str) -> str:
    return " ".join(strip_boxed(answer).split())

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    completions = [(c[0]['content'] if c else "") if isinstance(c, list) else c for c in completions]
    responses = [ANSWER_PATTERN.findall(completion) for completion in completions]

    predicted = responses[0][0] if responses[0] else "<no answer found>"
    print("###### Prompt:")
    print(prompts[0][0]["content"][1]["text"])
    print("\n###### True Answer:")
    print(standardize_answer(answer[0]))
    print("\n###### Predicted Answer:")
    print(standardize_answer(predicted))
    print("\n###### Full Response:")
    print(completions[0])
    print("\n================================================================================================================================\n")

    scores = []
    for response, true_answer in zip(responses, answer):
        if response == true_answer:
            scores.append(1.0)
            continue
        
        if len(response) != 1:
            scores.append(0.0)
            continue

        response = response[0]
        score = 0.0

        if response.startswith("\\boxed{") and response.endswith("}"):
            score += 0.45
        if standardize_answer(response) == standardize_answer(true_answer):
            score += 0.55

        scores.append(score)

    return scores

correctness_reward_func(
    [[{'role': 'user', 'content': [{'type': 'image', 'text': None}, {'type': 'text', 'text': "Question: In the figure, two equilateral triangles ABD and CBD each have side lengths of 1. Triangle ABD is translated to the right along the AC direction into position A'B'D', creating a shaded area. What is the perimeter of this shaded area? The above problem is with the following images:\n\nPlease first reason step by step using visual representations between <vthink> and </vthink>, then reason step by step in natural language between <think> and </think>, and finally provide your final answer in LaTeX using \\boxed{} between <answer> and </answer>"}]}]],
    ["<answer> Yes.\nLet's go! </answer>"],
    ["\\boxed{Yes.\nLet's go!}"],
)

# Training

In [ ]:
import math
from trl import GRPOConfig, GRPOTrainer

max_steps = math.ceil(len(train_dataset) / (MINI_BATCH_SIZE * GRAD_ACC_STEPS)) * NUM_EPOCHS
print("Calculated max steps:", max_steps)

training_args = GRPOConfig(
    # Training arguments
    seed = SEED,
    
    per_device_train_batch_size = MINI_BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACC_STEPS,
    num_generations = 4, # Decrease if out of memory
    
    max_prompt_length = 1024,
    max_completion_length = 2048,
    
    # max_steps = 50, # For testing
    max_steps = max_steps,
    # num_train_epochs = NUM_EPOCHS,
    # warmup_ratio=0.05,
    warmup_steps = WARMUP_STEPS,
    learning_rate = LR,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    max_grad_norm = 0.1,
    weight_decay = 0.01,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    
    # Logging arguments
    logging_strategy='steps',
    logging_steps = 1,
    log_completions = False,
    report_to=['tensorboard', 'wandb'],
    
    # Saving arguments
    save_strategy='steps',
    # save_steps=1, # For testing
    save_steps=10,
    # save_total_limit=5, # 1 best + 4 recent checkpoints. Warning: It doesn't work
    
    # With load_best_model_at_end=True, your save_strategy will be ignored and default to eval_strategy.
    # So you will find one checkpoint at the end of each epoch.
    # https://discuss.huggingface.co/t/trainer-not-saving-after-save-steps/5464
    # load_best_model_at_end=True, 

    output_dir=project_name,
    hub_model_id=hub_model_id,
    push_to_hub=True,

    hub_strategy='all_checkpoints',
    hub_always_push=True,

    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
)

trainer.train()

# Inference

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image = None
instruction = "Several slices of bread with side salads on a table."

messages = [
    {"role": "user", "content": [
        # {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

In [ ]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt = False)
outputs = model.generate(**inputs, 
                   streamer = text_streamer, 
                   max_new_tokens = 512,
                   use_cache = True, temperature = 1.5, min_p = 0.1)